# Morning class 27/08 — Extra practice 11 SOLUTIONS: scope and side effects   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q3 and Q8 both change something the caller owns. Only one of them is honest
about it.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 11 — Scope and side effects. Run this once.
THRESHOLD = 10
counter = 0

inventory = {"widget": 5, "gizmo": 0}
queue = ["a", "b", "c"]

print("THRESHOLD:", THRESHOLD, "| inventory:", inventory)

### Question 1

Global versus parameter. -> `False True` from both versions.

Same answers, and only one of the two can be reasoned about on its own.
`over(5)` depends on the state of the module; `over_explicit(5, 10)`
depends on nothing but its arguments.

The practical test: can you predict what `over(5)` returns without
scrolling? Worksheet 11 Q8 is what happens when the answer is no.

In [ ]:
def over(value):
    return value > THRESHOLD

def over_explicit(value, limit):
    return value > limit

print(over(5), over(50))
print(over_explicit(5, THRESHOLD), over_explicit(50, THRESHOLD))

### Question 2

Reading a global, assigning a local. -> `before: 0`, `inside: 1`, `after: 0`.

`local_counter = counter + 1` **reads** the global on the right and
**assigns** to a new local on the left. Both happen in one line and they
follow different rules.

Reading looks outward through LEGB. Assigning always creates a local,
unless you say `global`. That asymmetry is the whole of Python scope, and
Q9 is what happens when the same name is on both sides.

In [ ]:
print("before:", counter)

def bump():
    local_counter = counter + 1     # reads the global, assigns to a NEW local
    print("inside:", local_counter)

bump()
print("after: ", counter)

### Question 3

Mutating a dictionary argument. -> `before: {'widget': 5, 'gizmo': 0}`, **`after: {'widget': 8, 'gizmo': 0, 'sprocket': 7}`**.

`restock` returns nothing and changed the caller's dictionary anyway,
twice — one existing key topped up and one new key created.

Dicts are mutable, so the parameter `stock` is a second label on the
caller's object, exactly as with lists in worksheet 11 Q5. `stock[item] =
...` reaches into that object; it does not rebind the name.

At the call site `restock(inventory, "widget", 3)` gives no hint that
`inventory` is about to change. The name is the only warning — which is why
an imperative verb (`restock`, `add`, `update`) should mean "this mutates"
and a past participle (`restocked`) should mean "this returns a new one".

In [ ]:
print("before:", inventory)

def restock(stock, item, qty):
    stock[item] = stock.get(item, 0) + qty

restock(inventory, "widget", 3)
restock(inventory, "sprocket", 7)

print("after: ", inventory)

### Question 4

The non-mutating twin. -> `original: {'a': 1}`, `result:   {'a': 6}`.

`dict(stock)` makes a copy, the copy is modified, and the caller's
dictionary is untouched — so you can call this twice and get the same
answer twice.

It is a **shallow** copy. Nested dicts or lists inside would still be
shared, so this is safe for a flat structure and only half-safe for a
nested one.

The two names differ by three letters and that is deliberate: `restock`
versus `restocked`. Worksheet 11 Q6 makes the same point about lists, and
Python's own library follows the rule — `list.sort()` mutates, `sorted()`
returns.

In [ ]:
def restocked(stock, item, qty):
    """Return a NEW dict with `item` topped up. Leaves `stock` alone."""
    updated = dict(stock)
    updated[item] = updated.get(item, 0) + qty
    return updated

original = {"a": 1}
result = restocked(original, "a", 5)
print("original:", original)
print("result:  ", result)

### Question 5

Shadowing a builtin. -> `3`, `99`, `3`.

Between the second and third lines, `len` was an integer and `len(queue)`
would have raised `TypeError: 'int' object is not callable`.

That is the **B** in LEGB being covered up. Builtins are searched last, so
any name you create at module or function level hides the builtin of the
same name for the rest of that scope. `list`, `dict`, `sum`, `id`, `type`,
`max`, `filter` and `input` are all easy to shadow by accident.

The symptom is confusing because it appears **later and elsewhere** — some
unrelated function calls `len()` and gets a `TypeError` about integers.
`del len` removes your name and uncovers the builtin again, which is the
emergency fix; not choosing the name is the real one.

In [ ]:
print(len(queue))

len = 99
print(len)
# len(queue) here would raise TypeError: 'int' object is not callable

del len
print(len(queue))

### Question 6

A function that builds functions. -> `Hello, ana!`, `Hi, ana!`, `False`.

`make_greeting` returns the inner function itself, not a result — worksheet
09 Q5 again, with the function object crossing a `return`.

`hello` and `hi` are two **different** objects (`hello is hi` is `False`),
each carrying the `greeting` that was in scope when it was created. The
inner function keeps the enclosing variable alive after `make_greeting` has
finished. That arrangement is called a closure, and it is the **E** of LEGB
doing something useful.

This is how you build a family of related functions from one template —
and it is what `key=lambda ...` is quietly doing whenever the lambda
mentions a variable from around it.

In [ ]:
def make_greeting(greeting):
    def greet(name):
        return f"{greeting}, {name}!"
    return greet

hello = make_greeting("Hello")
hi = make_greeting("Hi")

print(hello("ana"))
print(hi("ana"))
print(hello is hi)

### Question 7

A function that consumes its input. -> `first call:  ['a', 'b', 'c']`, `queue now: []`, **`second call: []`**.

The first call returned everything and the second returned nothing, from
the same argument. `drain` empties the list it is given.

That is honest for a queue — consuming *is* the job — but it means the
function cannot be called twice, cannot be used in a retry, and cannot be
tested without rebuilding its input each time.

It is also the same shape as worksheet 12 Q10's exhausted iterator, and it
fails the same way: an empty result rather than an error.

If the caller might need the list afterwards, take a copy inside
(`items = list(items)`) or make the caller pass one.

In [ ]:
print("before:  ", queue)

def drain(items):
    taken = []
    while items:
        taken.append(items.pop(0))
    return taken

print("first call: ", drain(queue))
print("queue now:  ", queue)
print("second call:", drain(queue))

### Question 8

Pure versus impure. -> `6`, `6`, then **`6`, `12`**.

`total_pure([1, 2, 3])` gives 6 every time, forever, on any machine. Its
answer is a function of its arguments and nothing else.

`total_impure([1, 2, 3])` gives 6 the first time and 12 the second, from an
identical call. Its answer depends on how many times it has already been
called — history you cannot see at the call site.

That is the practical definition of "pure", and it is what makes a function
testable: you can assert the pure one's output without setting anything up
or cleaning anything down. The impure one needs `running` reset before
every test, and any test that forgets will pass or fail depending on the
order the tests ran in.

In [ ]:
running = 0

def total_pure(values):
    return sum(values)

def total_impure(values):
    global running
    running = running + sum(values)
    return running

print(total_pure([1, 2, 3]))
print(total_pure([1, 2, 3]))
print(total_impure([1, 2, 3]))
print(total_impure([1, 2, 3]))

### Question 9

Assigning to a global without saying so. -> `counter is 0`, then `UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value`.

Note the error: **`UnboundLocalError`, not `NameError`**. Python is not
saying `counter` does not exist — it is saying the *local* `counter` has no
value yet.

Because there is an assignment to `counter` somewhere in the body, Python
decides **at compile time** that the name is local for the whole function.
So `counter + 1` on the right-hand side looks up the local, which has not
been assigned yet, and the global is never consulted at all.

Q2 worked only because it assigned to a *different* name. Add
`counter = counter + 1` anywhere in that function and it breaks the same
way.

Two fixes, and they are not equal. `global counter` works and buys you
worksheet 11 Q3's problems. Taking the value as a parameter and returning
the new one is the answer.

In [ ]:
print("counter is", counter)

def increment():
    counter = counter + 1
    print(counter)

# This is SUPPOSED to raise: UnboundLocalError: cannot access local variable
# 'counter' where it is not associated with a value.
#
# Python decides at COMPILE time that `counter` is local to this function,
# because there is an assignment to it somewhere in the body. So the
# right-hand side looks up the LOCAL `counter`, which has no value yet -- the
# global is never consulted at all.
#
# Q2 worked because it assigned to a different name. The fixes are `global
# counter`, or better: take it as a parameter and return the new value.
increment()